<font color = #000000 >

## ANN手机价格分类案例

1. 构建数据集
2. 搭建神经网络
3. 模型训练
4. 模型测试

</font>

In [50]:
import sys
if "tf_env" not in sys.executable:
    print("/n环境配置错误!!!/n")
    print(sys.executable)
else:
    print("环境配置正常")

import numpy  as np
import pandas as pd
import matplotlib.pyplot as plt 
from matplotlib import rcParams # 字体配置,支持中文
rcParams['font.family'] = 'SimHei'
rcParams['axes.unicode_minus'] = False  # 解决负号不显示的问题

import torch
from torch.utils.data import TensorDataset  # 数据集对象
from torch.utils.data import DataLoader     # 数据加载器
import torch.nn as nn
import torch.optim as optim                 # 优化器
from sklearn.model_selection import train_test_split    # 训练集和测试集的划分
from torchsummary import summary            # 观察神经网络
import time 

环境配置正常


In [47]:
def tensor_print(t , num=0):
    print(f"{t} , type:{type(t)}" , end='')
    try:
        print(f' row:{t.shape[0]} , columns:{t.shape[1]} , last:{t.shape[-1]}')
    except Exception as e:
        print('t.shape:',t.shape)        
    if num != 0:
        print('-' * 15,end='')
        print(f' 第{num}项如上',end='')
        print('-' * 15)
    else:
        print('-'*30)

<font color = #000000 >

## 1. 构建数据集



</font>

In [48]:
def create_dataset():
    # 1. 数据集读取
    data = pd.read_csv('./data/手机价格预测.csv')
    print(data.shape) # (2000,21)
    # print(data.head())

    # 2. 获取x特征列和y标签列: x是手机的各个性能的特征,y是价格范围(0-3)
    x,y = data.iloc[:,:-1] , data.iloc[:,-1]

    # 3. 特征列转为浮点型
    x = x.astype(np.float32)

    # 4. 切分训练集和测试集
    # x,y: 特征 + 标签 , test_size: 测试集占比 , random_state: 随机种子 stratify: 样本分布(也就是参考y的类别进行抽取数据)
    x_train , x_test , y_trian , y_test = train_test_split(x,y,test_size=0.2,random_state=3,stratify=y)

    # 5. 数据封装为 张量数据集 , 思路: "数据 -> 张量Tensor -> 数据集TensorDataSet -> 数据加载器DataLoader"
    trian_dataset = TensorDataset(torch.tensor(x_train.values) , torch.tensor(y_trian.values))
    test_dataset  = TensorDataset(torch.tensor(x_test.values) , torch.tensor(y_test.values))
    # print(trian_dataset.tensors)

    # 6. 返回结果                         充当输入值: 20       充当输出值: 4
    return trian_dataset, test_dataset , x_train.shape[1] , len(np.unique(y))

if __name__ == '__main__':
    trian_dataset , test_dataset , input_dim , output_dim = create_dataset()
    print(f'训练集 数据集对象: {trian_dataset.tensors}')
    print(f'测试集 数据集对象: {test_dataset.tensors}')
    print(f"输入特征数:{input_dim}")
    print(f"输出特征数:{output_dim}")

(2000, 21)
训练集 数据集对象: (tensor([[9.5900e+02, 1.0000e+00, 2.6000e+00,  ..., 1.0000e+00, 0.0000e+00,
         0.0000e+00],
        [1.7240e+03, 0.0000e+00, 2.0000e+00,  ..., 1.0000e+00, 1.0000e+00,
         0.0000e+00],
        [1.4560e+03, 1.0000e+00, 5.0000e-01,  ..., 1.0000e+00, 0.0000e+00,
         1.0000e+00],
        ...,
        [8.2300e+02, 1.0000e+00, 2.7000e+00,  ..., 1.0000e+00, 1.0000e+00,
         1.0000e+00],
        [6.8300e+02, 1.0000e+00, 2.1000e+00,  ..., 0.0000e+00, 0.0000e+00,
         0.0000e+00],
        [1.4690e+03, 0.0000e+00, 5.0000e-01,  ..., 1.0000e+00, 0.0000e+00,
         0.0000e+00]]), tensor([3, 3, 1,  ..., 0, 1, 2]))
测试集 数据集对象: (tensor([[1.5890e+03, 0.0000e+00, 2.6000e+00,  ..., 1.0000e+00, 1.0000e+00,
         0.0000e+00],
        [6.0600e+02, 0.0000e+00, 2.5000e+00,  ..., 1.0000e+00, 0.0000e+00,
         0.0000e+00],
        [1.9630e+03, 1.0000e+00, 1.0000e+00,  ..., 1.0000e+00, 0.0000e+00,
         1.0000e+00],
        ...,
        [6.4500e+02, 0.0000e+0

<font color = #000000 >

## 2. 构建分类神经网络模型

+ 网络由3个线性层构建,使用ReLU激活函数

+ 第1层: in 20 out 128
+ 第2层: in 128 out 256
+ 第3层: in 256 out 4

</font>

In [51]:
# 搭建神经网络
class PhonePriceModel(nn.Module):

    # 1. 在Init方法中完成初始化
    def __init__(self, input_dim , output_dim):
        '''
        核心步骤:
            1. 父类成员初始化
            2. 搭建神经网络:隐藏层 + 输出层
            3. 对隐藏层进行w/b初始化(可以不做)
        '''
        # 1-1 父类成员初始化
        super().__init__()
        # 1-2 搭建神经网络:隐藏层 + 输出层
        self.linear1 = nn.Linear(in_features=input_dim ,out_features=128)
        self.linear2 = nn.Linear(in_features=128,out_features=256)
        self.output  = nn.Linear(in_features=256,out_features=output_dim)

        # # 1-3 对隐藏层进行w/b初始化
        # # 隐藏层1
        # nn.init.xavier_normal_(self.linear1.weight)
        # nn.init.zeros_(self.linear1.bias)
        # # 隐藏层2
        # nn.init.xavier_normal_(self.linear2.weight)
        # nn.init.zeros_(self.linear2.bias)
        # # 输出层不需要w/b初始化

    # 2. 前向传播: 输入层 -> 隐藏层 -> 输出层
    def forward(self , x):
        '''
        核心步骤:
            1. 对各个层进行配置: 加权求和 + 激活函数(Sigmoid)
            2. 返回计算结果(前向传播结果)
        '''

        # 第1层 隐藏层计算: 加权求和 + 激活函数(ReLU)
        x = torch.relu(self.linear1(x))   # 加权求和 + 激活函数

        # 第2层 隐藏层计算: 加权求和 + 激活函数(ReLU)
        x = torch.relu(self.linear2(x))   # 加权求和 + 激活函数

        # 第3层 输出层计算: 加权求和 + 激活函数(Softmax,因为是多分类,所以后续使用多分类交叉熵,所以输出层不需要激活函数初始化)
        x = self.output(x)

        # 返回
        return x 

if __name__ == '__main__':
    # 1. 构建数据集
    trian_dataset , test_dataset , input_dim , output_dim = create_dataset()

    # 2. 构建神经网络模型
    model = PhonePriceModel(input_dim , output_dim)

    # 3. 计算模型参数
    summary(model , input_size=(16,input_dim))

(2000, 21)
----------------------------------------------------------------
        Layer (type)               Output Shape         Param #
            Linear-1              [-1, 16, 128]           2,688
            Linear-2              [-1, 16, 256]          33,024
            Linear-3                [-1, 16, 4]           1,028
Total params: 36,740
Trainable params: 36,740
Non-trainable params: 0
----------------------------------------------------------------
Input size (MB): 0.00
Forward/backward pass size (MB): 0.05
Params size (MB): 0.14
Estimated Total Size (MB): 0.19
----------------------------------------------------------------


<font color = #000000 >

## 3. 模型训练


</font>

In [65]:
# 3. 模型训练
def trian(trian_dataset , input_dim , output_dim):
    # 1. 创建数据加载器: 数据 -> 张亮 -> 数据集(当前传入的参数) -> 数据加载器
    # trian_dataset: 训练数据集 batch_size:每轮批次 shuffle: 是否打乱
    trian_loader = DataLoader(trian_dataset , batch_size=16 , shuffle=True)

    # 2. 创建神经网络模型
    model = PhonePriceModel(input_dim , output_dim)

    # 3. 定义损失函数,因为是多分类,所以使用多分类交叉熵损失函数
    criterion = nn.CrossEntropyLoss()

    # 4. 创建优化器对象
    optimizer = optim.SGD(model.parameters() , lr=0.001) # 待优化

    # 5. 模型训练
    # 5-1 定义函数,记录训练的总轮次
    epochs = 50
    # 5-2 开始每轮的训练
    for epoch in range(epochs):
        # 5-2-1 定义变量,记录每次训练的损失值和训练批次数
        total_loss , batch_num = 0.0 , 0
        # 5-2-2 定义变量,表示训练开始的时间
        start = time.time()
        # 5-2-3 开始本轮次的各个批次的训练
        for x, y in trian_loader:
            # 1. 切换模型模式(状态:训练or测试)
            model.train()   # 训练模式
            # model.eval()    # 测试模式

            # 2. 模型预测
            y_pred = model(x)

            # 3. 计算损失
            loss = criterion(y_pred , y)

            # 4. 三步骤
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            # 5. 累积损失值
            total_loss += loss.item()   # 把本轮的本批次的平均损失累积起来(一轮加16次)
            batch_num += 1
        
        # 5-2-4 本轮训练结束,打印训练信息
        print(f'epoch:{epoch+1} , loss:{total_loss / batch_num :.4f} , time:{time.time() - start :.2f}s')
    
    # 6. 保存模型(参数)
    torch.save(model.state_dict() , './model/phone.pth')    # 后缀名: .pth


if __name__ == '__main__':
    # 1. 构建数据集
    trian_dataset , test_dataset , input_dim , output_dim = create_dataset()

    # 2. 构建神经网络模型
    model = PhonePriceModel(input_dim , output_dim)

    # 3. 计算模型参数
    summary(model , input_size=(16,input_dim))

    # 4. 模型训练
    trian(trian_dataset , input_dim , output_dim)

(2000, 21)
----------------------------------------------------------------
        Layer (type)               Output Shape         Param #
            Linear-1              [-1, 16, 128]           2,688
            Linear-2              [-1, 16, 256]          33,024
            Linear-3                [-1, 16, 4]           1,028
Total params: 36,740
Trainable params: 36,740
Non-trainable params: 0
----------------------------------------------------------------
Input size (MB): 0.00
Forward/backward pass size (MB): 0.05
Params size (MB): 0.14
Estimated Total Size (MB): 0.19
----------------------------------------------------------------
epoch:1 , loss:22.7040 , time:0.07s
epoch:2 , loss:1.1723 , time:0.08s
epoch:3 , loss:1.0921 , time:0.08s
epoch:4 , loss:1.1100 , time:0.07s
epoch:5 , loss:1.0296 , time:0.06s
epoch:6 , loss:0.9980 , time:0.06s
epoch:7 , loss:1.0226 , time:0.06s
epoch:8 , loss:0.9545 , time:0.06s
epoch:9 , loss:0.9734 , time:0.07s
epoch:10 , loss:0.9641 , time:0.06s
e

<font color = #000000 >

## 4. 模型测试

+ 一般不要定义函数名字叫test

</font>

In [73]:
def evaluate(test_dataset , input_dim , output_dim):
    # 1. 创建神经网络分类对象
    model = PhonePriceModel(input_dim , output_dim)

    # 2. 加载模型参数
    model.load_state_dict(torch.load('./model/phone.pth'))

    # 3. 创建测试机的数据加载器对象
    test_loader = DataLoader(test_dataset , batch_size=16 , shuffle=False)

    # 4. 定义变量,记录预测正确的样本个数
    correct = 0     # 总共400个

    # 5. 从数据加载器中,获取到每批次的数据
    for x , y in test_loader:
        # 5-1 切换为测试模式
        model.eval()

        # 5-2 模型预测
        y_pred = model(x)   # 得到的结果没有进行softmax加权处理,所以甚至有负数,但是一旦进行softmax就是真正的概率,不过直接求max即可
        # print(f'y_pred:{y_pred}')
        
        # 5-3 根据加权求和,得到类别,使用argmax()获取最大值对应的下标,就是类别
        y_pred = torch.argmax(y_pred, dim=1)    # 找到每1行的最大值

        print(f'y_pred:\t{y_pred}')
        print(f"y真=\t{y}")

        # 5-4 统计预测正确的样本个数
        print(y_pred == y)
        print((y_pred == y).sum())

        correct += (y_pred == y).sum()
    # 6. 打印准确率即可
    print(f"准确值:{correct / len(test_dataset):.4f}")


if __name__ == '__main__':
    # 1. 构建数据集
    trian_dataset , test_dataset , input_dim , output_dim = create_dataset()

    # 2. 构建神经网络模型
    model = PhonePriceModel(input_dim , output_dim)

    # 3. 计算模型参数
    summary(model , input_size=(16,input_dim))

    # 4. 模型训练
    # trian(trian_dataset , input_dim , output_dim)

    # 5. 模型测试
    evaluate(test_dataset , input_dim,  output_dim)

(2000, 21)
----------------------------------------------------------------
        Layer (type)               Output Shape         Param #
            Linear-1              [-1, 16, 128]           2,688
            Linear-2              [-1, 16, 256]          33,024
            Linear-3                [-1, 16, 4]           1,028
Total params: 36,740
Trainable params: 36,740
Non-trainable params: 0
----------------------------------------------------------------
Input size (MB): 0.00
Forward/backward pass size (MB): 0.05
Params size (MB): 0.14
Estimated Total Size (MB): 0.19
----------------------------------------------------------------
y_pred:	tensor([3, 1, 1, 2, 0, 1, 1, 1, 0, 2, 3, 2, 3, 3, 2, 3])
y真=	tensor([2, 0, 1, 2, 0, 1, 0, 0, 0, 1, 3, 2, 3, 3, 1, 3])
tensor([False, False,  True,  True,  True,  True, False, False,  True, False,
         True,  True,  True,  True, False,  True])
tensor(10)
y_pred:	tensor([2, 3, 0, 2, 2, 3, 0, 0, 3, 0, 1, 2, 0, 2, 1, 1])
y真=	tensor([2, 2, 0, 2

<font color = #000000 >

## 5. 优化思路

+ 优化方法:从SGD->Adam
+ 学习率从0.001 -> 0.0001
+ 标准化,归一化
+ 增加网络的深度,每层的神经元数量
+ 调整训练的轮数


</font>

In [74]:
import torch
import torch.nn as nn
import pandas as pd
from sklearn.model_selection import train_test_split
from torch.utils.data import TensorDataset
from torch.utils.data import DataLoader
import torch.optim as optim
import numpy as np
import time
from sklearn.preprocessing import StandardScaler


# 构建数据集
def create_dataset():
	# 使用pandas读取数据
	data = pd.read_csv('./data/手机价格预测.csv')
	# 特征值和目标值
	x, y = data.iloc[:, :-1], data.iloc[:, -1]
	# 类型转换：特征值，目标值
	x = x.astype(np.float32)
	y = y.astype(np.int64)
	# 数据集划分
	x_train, x_valid, y_train, y_valid = train_test_split(x, y, train_size=0.8, random_state=88, stratify=y)
	# 优化①:数据标准化
	transfer = StandardScaler()
	x_train = transfer.fit_transform(x_train)
	x_valid = transfer.transform(x_valid)
	# 构建数据集,转换为pytorch的形式
	train_dataset = TensorDataset(torch.from_numpy(x_train), torch.tensor(y_train.values))
	valid_dataset = TensorDataset(torch.from_numpy(x_valid), torch.tensor(y_valid.values))
	# 返回结果
	return train_dataset, valid_dataset, x_train.shape[1], len(np.unique(y))


# 构建网络模型
class PhonePriceModel(nn.Module):

	def __init__(self, input_dim, output_dim):
		super(PhonePriceModel, self).__init__()
		# 优化②:增加网络深度
		# 1. 第一层: 输入为维度为 20, 输出维度为: 128
		self.linear1 = nn.Linear(input_dim, 128)
		# 2. 第二层: 输入为维度为 128, 输出维度为: 256
		self.linear2 = nn.Linear(128, 256)
		# 3. 第三层: 输入为维度为 256, 输出维度为: 512
		self.linear3 = nn.Linear(256, 512)
		# 4. 第四层: 输入为维度为 512, 输出维度为: 128
		self.linear4 = nn.Linear(512, 128)
		# 5. 输出层: 输入为维度为 128, 输出维度为: 4
		self.linear5 = nn.Linear(128, output_dim)

	def forward(self, x):
		# 前向传播过程
		x = torch.relu(self.linear1(x))
		x = torch.relu(self.linear2(x))
		x = torch.relu(self.linear3(x))
		x = torch.relu(self.linear4(x))
		# 后续CrossEntropyLoss损失函数中包含softmax过程, 所以当前步骤不进行softmax操作
		output = self.linear5(x)
		# 获取数据结果
		return output


# 编写训练函数
def train(train_dataset, input_dim, class_num):
	# 固定随机数种子
	torch.manual_seed(0)
	# 初始化数据加载器
	dataloader = DataLoader(train_dataset, shuffle=True, batch_size=8)
	# 初始化模型
	model = PhonePriceModel(input_dim, class_num)
	# 损失函数 CrossEntropyLoss = softmax + 损失计算
	criterion = nn.CrossEntropyLoss()
	# 优化③:使用Adam优化方法, 优化④:学习率变为1e-4
	optimizer = optim.Adam(model.parameters(), lr=1e-4)
	# 遍历每个轮次的数据
	num_epoch = 50
	for epoch_idx in range(num_epoch):
		# 训练时间
		start = time.time()
		# 计算损失
		total_loss = 0.0
		total_num = 0
		# 遍历每个batch数据进行处理
		for x, y in dataloader:
			model.train()
			output = model(x)
			# 计算损失
			loss = criterion(output, y)
			# 梯度清零
			optimizer.zero_grad()
			# 反向传播
			loss.backward()
			# 参数更新
			optimizer.step()
			# 损失计算
			total_num += len(y)
			total_loss += loss.item() * len(y)
		# 打印损失变换结果
		print('epoch: %4s loss: %.2f, time: %.2fs' %
			  (epoch_idx + 1, total_loss / total_num, time.time() - start))
	# 模型保存
	torch.save(model.state_dict(), './model/phone-price-model2.pth')


def evaluate(valid_dataset, input_dim, class_num):
	# 加载模型和训练好的网络参数
	model = PhonePriceModel(input_dim, class_num)
	# load_state_dict:将加载的参数字典应用到模型上
	# load:加载用来保存模型参数的文件
	model.load_state_dict(torch.load('./model/phone-price-model2.pth'))
	# 构建加载器
	dataloader = DataLoader(valid_dataset, batch_size=8, shuffle=False)
	# 评估测试集
	correct = 0
	# 遍历测试集中的数据
	for x, y in dataloader:
		# 将其送入网络中
		# model.eval()
		output = model(x)
		# 获取预测类别结果
		y_pred = torch.argmax(output, dim=1)
		# 获取预测正确的个数
		correct += (y_pred == y).sum()
	# 求预测精度
	print('Acc: %.5f' % (correct / len(valid_dataset)))


if __name__ == '__main__':
	train_dataset, valid_dataset, input_dim, class_num = create_dataset()
	train(train_dataset, input_dim, class_num)
	evaluate(valid_dataset, input_dim, class_num)

epoch:    1 loss: 1.23, time: 0.41s
epoch:    2 loss: 0.52, time: 0.57s
epoch:    3 loss: 0.29, time: 0.48s
epoch:    4 loss: 0.23, time: 0.82s
epoch:    5 loss: 0.19, time: 1.05s
epoch:    6 loss: 0.16, time: 0.47s
epoch:    7 loss: 0.14, time: 0.45s
epoch:    8 loss: 0.12, time: 0.51s
epoch:    9 loss: 0.10, time: 0.44s
epoch:   10 loss: 0.08, time: 0.48s
epoch:   11 loss: 0.08, time: 0.45s
epoch:   12 loss: 0.07, time: 0.44s
epoch:   13 loss: 0.06, time: 0.47s
epoch:   14 loss: 0.05, time: 0.49s
epoch:   15 loss: 0.04, time: 0.49s
epoch:   16 loss: 0.04, time: 0.46s
epoch:   17 loss: 0.03, time: 0.44s
epoch:   18 loss: 0.02, time: 0.53s
epoch:   19 loss: 0.02, time: 0.48s
epoch:   20 loss: 0.01, time: 0.55s
epoch:   21 loss: 0.02, time: 0.75s
epoch:   22 loss: 0.01, time: 0.75s
epoch:   23 loss: 0.01, time: 0.73s
epoch:   24 loss: 0.01, time: 0.83s
epoch:   25 loss: 0.01, time: 0.96s
epoch:   26 loss: 0.00, time: 0.73s
epoch:   27 loss: 0.00, time: 0.48s
epoch:   28 loss: 0.00, time